In [8]:
!pip install -q transformers==4.41.0
!pip install -q peft==0.10.0
!pip install -q datasets
!pip install -q accelerate
!pip install -q soundfile
print("Kutubxonalar yukklandi!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.4 MB/s eta 0:00:00
Kutubxonalar yukklandi!


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

device="cuda" if torch.cuda.is_available() else "cpu"
processor=WhisperProcessor.from_pretrained(
    "/content/drive/MyDrive/whisper-lora-finetuned"
)
model=WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
)
model=model.to(device)
print(f"Model yuklandi! Device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model yuklandi! Device: cuda


In [17]:
from datasets import load_dataset, Audio

test_data=load_dataset(
    "google/fleurs",
    "en_us",
    split="test"
)
sample=test_data[0]

inputs=processor(
   sample["audio"]["array"],
   sampling_rate=16000,
   return_tensor="pt"
)
input_features=torch.from_numpy(inputs.input_features).to(device)

model.eval()
with torch.no_grad():
  predicted_ids=model.generate(input_features)

transcription=processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]
print(f"Model aytdi: {transcription}")
print(f"Asl matn: {sample['transcription']}")

Model aytdi:  However, due to the slow communication channels, styles in the West could lag behind by 25 to 30 years.
Asl matn: however due to the slow communication channels styles in the west could lag behind by 25 to 30 year


In [21]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.0 MB/s eta 0:00:00


In [22]:
import jiwer

def normalize(text):
  return text.lower().strip().rstrip(".")

reference=normalize(sample["transcription"])
hypothesis=normalize(transcription)
wer=jiwer.wer(reference, hypothesis)
print(f"WER: {wer * 100:.2f} %")

if wer==0:
  print("Mukammal! 0% xatolik")
elif wer< 0.1:
  print("Juda yaxshi")
else:
  print("Yaxshilash kerak")

WER: 15.79 %
Yaxshilash kerak


In [23]:
# ============================================
# CELL 18: Gradio UI yasash
# Foydalanuvchi mikrofon orqali gapirib,
# model matnga o'giradi
# ============================================

# gradio — web interfeys yaratish kutubxonasi
!pip install -q gradio

import gradio as gr
import torch

# transcribe() — asosiy funksiya
# audio berilsa → matn qaytaradi
def transcribe(audio):

    # Audio faylni o'qish
    import librosa
    audio_array, sr = librosa.load(audio, sr=16000)
    # librosa.load() — audio faylni numpy arrayga o'giradi
    # sr=16000 — Whisper 16000 Hz talab qiladi

    # Audio → model tushunadigan format
    inputs = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)
    # processor() — audio signalni model uchun tayyorlaydi
    # return_tensors="pt" — PyTorch tensor formatida
    # .to(device) — GPU ga yuborish

    # Model matn chiqarsin
    model.eval()
    # model.eval() — modelni test rejimiga o'tkazish
    # Training rejimida emas, faqat bashorat qilish

    with torch.no_grad():
        # torch.no_grad() — gradient hisoblama
        # Tezroq ishlaydi, xotira tejaydi
        predicted_ids = model.generate(inputs)
        # model.generate() — model matn tokenlarini yaratadi

    # Tokenlarni matnga o'girish
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
        # skip_special_tokens — <|en|> kabi maxsus tokenlarni o'chirish
    )[0]

    return transcription

# Gradio interfeys yaratish
app = gr.Interface(
    fn=transcribe,          # Qaysi funksiyani chaqirish
    inputs=gr.Audio(
        type="filepath"     # Audio fayl sifatida qabul qilish
    ),
    outputs=gr.Textbox(
        label="Matn"        # Natija ko'rsatiladigan joy
    ),
    title="🎤 Whisper STT",
    description="Audio yuboring → matn chiqadi!"
)

# Ilovani ishga tushirish
app.launch(share=True)
# share=True — internet orqali ochiq link beradi

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://372451bca8b9f72e9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:

from huggingface_hub import HfApi, login, create_repo

# Login
login(token="hf_........")

api = HfApi()

# 1-qadam: Repo yaratish
# exists_ok=True — agar repo bor bo'lsa xato bermaydi
create_repo(
    repo_id="dilnoza-nugmanova/whisper-lora-finetuned",
    repo_type="model",
    exist_ok=True
)
print("✅ Repo yaratildi!")

# 2-qadam: Modelni yuklash
api.upload_folder(
    folder_path="/content/drive/MyDrive/whisper-lora-finetuned",
    repo_id="dilnoza-nugmanova/whisper-lora-finetuned",
    repo_type="model"
)

print("✅ Model yuklandi!")
print("🔗 huggingface.co/dilnoza-nugmanova/whisper-lora-finetuned")

✅ Repo yaratildi!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model yuklandi!
🔗 huggingface.co/dilnoza-nugmanova/whisper-lora-finetuned


In [26]:
from huggingface_hub import HfApi, create_repo
import os

# Space yaratish
# space_sdk="gradio" — Gradio asosida ishlaydi
create_repo(
    repo_id="dilnoza-nugmanova/whisper-lora-stt",
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)
print("✅ Space yaratildi!")

✅ Space yaratildi!


In [27]:
# ============================================
# CELL 21: app.py yaratish
# HuggingFace Space uchun asosiy fayl
# ============================================

app_code = '''
import gradio as gr
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
import librosa

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model yuklash
processor = WhisperProcessor.from_pretrained(
    "dilnoza-nugmanova/whisper-lora-finetuned"
)
base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
)
model = PeftModel.from_pretrained(
    base_model,
    "dilnoza-nugmanova/whisper-lora-finetuned"
)
model = model.to(device)
model.eval()

def transcribe(audio):
    # Audio yuklash
    audio_array, sr = librosa.load(audio, sr=16000)

    # Tayyorlash
    inputs = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    # Matn chiqarish
    with torch.no_grad():
        predicted_ids = model.generate(inputs)

    text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return text

# Gradio UI
app = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(type="filepath", label="Audio yuboring"),
    outputs=gr.Textbox(label="Matn"),
    title="🎤 Whisper LoRA STT",
    description="Fine-tuned Whisper modeli — audio → matn!"
)

app.launch()
'''

# Faylga yozish
with open("/content/app.py", "w") as f:
    f.write(app_code)
print("✅ app.py yaratildi!")

✅ app.py yaratildi!


In [29]:
req = """transformers
peft
gradio
librosa
torch
accelerate
soundfile
"""

with open("/content/requirements.txt", "w") as f:
    f.write(req)

# Space ga qayta yuklash
api.upload_file(
    path_or_fileobj="/content/requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="dilnoza-nugmanova/whisper-lora-stt",
    repo_type="space"
)

print("✅ requirements.txt yangilandi!")
print("⏳ Space qayta build bo'ladi — 3-5 daqiqa kuting")

✅ requirements.txt yangilandi!
⏳ Space qayta build bo'ladi — 3-5 daqiqa kuting
